# ML-09 Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Two jobs in this notebook. First, read two findings from the FlyRank March 2026 paper and write down
the methodology question I would ask about each. Second, turn the same reading on my own Week 5 model:
put it under a stricter split, audit the feature set for leakage, look at pages it gets wrong, and
rewrite my own sentences that went further than the evidence.

My lane is Refresh, or Content Opportunity Scoring. Week 4 was a hand written rule, Week 5 was a
random forest on within month momentum validated with client grouped folds. This week I try to break
both of them.

## 1. Two paper findings + my methodology questions

Framing first, because it matters. The paper states its own evidence standard on page 4: direct
aggregate comparisons lead, machine learning is exploratory appendix material that does not override
direct evidence, and external SEO beliefs are treated as hypotheses. It keeps a whole page of results
that weakened or reversed (page 18), and it flags its own unstable buckets in place, for example the
`361+` freshness ratio of 283:1 that rests on a single declining page (page 9) and the survivor bias
in the `365+ x 361+` cell of the age freshness matrix (page 14). That is more self disclosure than
most public marketing research carries. My questions below are the next turn of the same screw, not a
complaint that the screw is missing.

### Finding #4, The Freshness Multiplier (page 9)

> "365+ day content that was refreshed within 30 days shows **3.2x health boost** (from 10.7 to 34.5)
> and **57x more impressions** (from 71 to 4039). In this portfolio, refresh timing is one of the
> strongest measured levers available."

**My methodology question: who chose which pages got refreshed, and can the design separate the
refreshing from the choosing?**

Nobody refreshes pages at random. A team picks them, and the realistic selection rule is "pages that
still matter": ones with existing demand, strategic value, or a ranking worth defending. The
comparison group, old pages not refreshed in the last 30 days, is therefore partly made of pages
somebody looked at and decided to abandon. A 57x impressions gap between "chosen" and "not chosen"
is measuring the choice and the refresh together, and on this design there is no way to say how the
57 splits between them. The paper's own nuanced list already says the right thing, "refreshing strong
pages works far better than refreshing thin pages that were weak to begin with" (page 18), which is
the selection effect described from the other side.

Two smaller things in the same finding. The health score is defined on page 5 as impressions 30 points
plus position 30 plus CTR 20 plus scroll depth 20, so impressions are a component of health. The 3.2x
health boost and the 57x impression boost are then not two independent confirmations, they overlap by
construction, and quoting them as separate proof points reads as more evidence than one measurement
can carry. And both refresh recency and the 90 day performance window come from the same snapshot, so
I cannot tell from the page whether the impressions counted fall after the refresh date or straddle it.

**How I would make it stronger, not smaller.** Match each refreshed page to an unrefreshed page of
similar age, pre period impressions, and position, then compare only within matched pairs. Report the
pre refresh impressions of both groups in the same table, so a reader can see the starting gap. Count
impressions strictly in the window after the refresh date. Even a rough matched cut would let the
sentence move from "refresh timing is a strong lever" to "among pages that started at comparable
visibility, refreshed pages ended the window at N times the impressions", which is a smaller number
and a much harder one to argue with.

### Finding #2, The Content Performance Curve (page 7)

> "Content enters a **growth phase** during its first 90 days ... It hits **peak performance at 61-90
> days** (health score 33.1). A **maturation plateau** follows from 91-180 days ... Then comes the
> **decay cliff at 271-365 days** (health drops to 14)."

**My methodology question: this is one snapshot across many pages, so what licenses reading it as the
life of a page over time?**

The chart compares different pages that happen to be different ages today. The prose reads it as a
sequence a single page passes through, which is a claim about change within a page. A cross section
cannot separate the age effect from the cohort effect: pages created 400 days ago were written by a
different team, to a different brief, in a different market, than pages created 40 days ago. Whatever
changed about the publishing process over that period shows up in this chart looking exactly like
ageing.

Survivorship pushes the same way. Extended cuts are drawn from an active content subset with
`impressions_90d > 0 and sessions_90d > 0` (page 4). Old pages that died or were deleted are missing
from the old buckets, so the surviving old pages are the ones that kept working. The paper names this
bias for the age freshness matrix on page 14 but the lifecycle curve on page 7 carries it too, and the
`365+` rebound to 25.1 is exactly where it would bite hardest.

There is also a definitional floor at the young end. The `0-7` day bucket scores 7 health, and health
is 30 percent impressions. A page published five days ago has barely been crawled, so it cannot have
impressions yet. Part of "content enters a growth phase" is indexing latency inside the metric
definition rather than content getting better.

**How I would make it stronger.** The paper already holds a monthly series from 2025-10 to 2026-03
(page 17). Fix a single creation cohort, follow those same page ids across those months, and plot
health against months since publication. That turns the snapshot into an actual lifecycle and it costs
no new data. Say the survivor filter in the same sentence as the finding, and consider a health
variant without the impressions component for the youngest buckets so the floor is not mistaken for
a growth phase.

The cell below puts numbers from my own slice behind the second question, because I can test that one
directly.

In [1]:
# Setup. Token comes from the environment, Colab Secrets, or a prompt. It is NEVER
# written into this notebook, which lives in a public repo.
import os, json, duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 220)

def _hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        import getpass
        return getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{_hf_token()}')")

REL  = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIMC = f"{REL}/dim_content.parquet"
MONTHS = ["2026-03", "2026-04", "2026-05"]     # two feature months, two label months, one overlap

def _out_dir():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, ".git")) or os.path.isdir(os.path.join(d, "work", "outputs")):
            return os.path.join(d, "work", "outputs")
        d = os.path.dirname(d)
    return "work/outputs"
OUT = _out_dir(); os.makedirs(OUT, exist_ok=True)
print("connected; outputs ->", OUT)

connected; outputs -> /home/zuko/ml-internship/work/outputs


In [2]:
# One pass over three months. Week 5 used a single month pair, March features and an April label.
# To test the model forward in time I need a second, later pair on the identical recipe:
# April features and a May label. Same SQL, same half month split rule, different dates.
paths = ", ".join(f"'{FACT}/month={m}/*.parquet'" for m in MONTHS)

def month_cols(pre, lo, mid, hi):
    """Feature aggregates for one month, split at the 15th. All inside that month only."""
    return f"""
       SUM(CASE WHEN report_date BETWEEN DATE '{lo}' AND DATE '{hi}' THEN gsc_impressions  ELSE 0 END) AS {pre}_impr,
       SUM(CASE WHEN report_date BETWEEN DATE '{lo}' AND DATE '{hi}' THEN gsc_clicks       ELSE 0 END) AS {pre}_clicks,
       SUM(CASE WHEN report_date BETWEEN DATE '{lo}' AND DATE '{hi}' THEN gsc_sum_position ELSE 0 END) AS {pre}_sumpos,
       COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{lo}' AND DATE '{hi}' AND gsc_impressions > 0) AS {pre}_days,
       SUM(CASE WHEN report_date BETWEEN DATE '{lo}' AND DATE '{mid}' THEN gsc_impressions ELSE 0 END) AS {pre}_h1_impr,
       SUM(CASE WHEN report_date >  DATE '{mid}' AND report_date <= DATE '{hi}' THEN gsc_impressions ELSE 0 END) AS {pre}_h2_impr,
       SUM(CASE WHEN report_date BETWEEN DATE '{lo}' AND DATE '{mid}' THEN gsc_clicks      ELSE 0 END) AS {pre}_h1_clicks,
       SUM(CASE WHEN report_date >  DATE '{mid}' AND report_date <= DATE '{hi}' THEN gsc_clicks      ELSE 0 END) AS {pre}_h2_clicks"""

w = con.sql(f"""
    SELECT content_hash_id,
      {month_cols('mar', '2026-03-01', '2026-03-15', '2026-03-31')},
      {month_cols('apr', '2026-04-01', '2026-04-15', '2026-04-30')},
      SUM(CASE WHEN report_date BETWEEN DATE '2026-05-01' AND DATE '2026-05-31'
               THEN gsc_impressions ELSE 0 END) AS may_impr
    FROM read_parquet([{paths}])
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    ORDER BY content_hash_id   -- a hash aggregate returns rows in no fixed order, and row order
                               -- reaches the bootstrap sampler, so without this the third decimal
                               -- of every score moves between runs. Sorted, the notebook reproduces.
""").df()

dc = con.sql(f"SELECT content_hash_id, client_hash_id, content_created_date FROM read_parquet('{DIMC}')").df()
dc["content_created_date"] = pd.to_datetime(dc["content_created_date"])

def make_panel(pre, next_col, decision_day):
    """One panel: features from month `pre`, label from `next_col`. Neutral column names so the
    identical model can be fitted on either month pair. Same ten features as ML-08."""
    p = pd.DataFrame({
        "content_hash_id": w["content_hash_id"],
        "impr":   w[f"{pre}_impr"],       "clicks":    w[f"{pre}_clicks"],
        "sumpos": w[f"{pre}_sumpos"],     "days_with_impr": w[f"{pre}_days"],
        "h1_impr":   w[f"{pre}_h1_impr"], "h2_impr":   w[f"{pre}_h2_impr"],
        "h1_clicks": w[f"{pre}_h1_clicks"],"h2_clicks": w[f"{pre}_h2_clicks"],
        "next_impr": w[next_col],
    })
    p = p[p["impr"] > 0].copy()
    p["ctr"]          = 100.0 * p["clicks"] / p["impr"]
    p["avg_position"] = p["sumpos"] / p["impr"]
    p["mom_impr"]     = (p["h2_impr"] + 1) / (p["h1_impr"] + 1)
    p["mom_clicks"]   = (p["h2_clicks"] + 1) / (p["h1_clicks"] + 1)
    p = p.merge(dc, on="content_hash_id", how="left")
    p["age_days"] = (pd.Timestamp(decision_day) - p["content_created_date"]).dt.days
    p = p[p["age_days"].notna()].copy()
    p["decline"] = (p["next_impr"] < 0.8 * p["impr"]).astype(int)
    return p.reset_index(drop=True)

P1 = make_panel("mar", "apr_impr", "2026-03-31")   # the Week 5 panel
P2 = make_panel("apr", "may_impr", "2026-04-30")   # the same recipe one month later

print("PANEL 1  features 2026-03, label 2026-04, decision day 2026-03-31")
print(f"  pages {len(P1):,}   clients {P1['client_hash_id'].nunique()}   base decline rate {P1['decline'].mean():.4f}")
print("PANEL 2  features 2026-04, label 2026-05, decision day 2026-04-30")
print(f"  pages {len(P2):,}   clients {P2['client_hash_id'].nunique()}   base decline rate {P2['decline'].mean():.4f}")

# Panel 1 must be the exact frame ML-08 used, otherwise "before and after" compares two things.
assert len(P1) == 176738, f"panel 1 no longer matches ML-08 ({len(P1)} rows)"
assert round(P1["decline"].mean(), 4) == 0.5319, "panel 1 base rate no longer matches ML-08"
print("\ncheck: panel 1 reproduces the ML-08 frame exactly, 176,738 pages at base rate 0.5319.")

# The halves must still reconcile to the month total in BOTH panels.
for nm, pre in (("panel 1", "mar"), ("panel 2", "apr")):
    gap = (w[f"{pre}_h1_impr"] + w[f"{pre}_h2_impr"] - w[f"{pre}_impr"]).abs().max()
    print(f"check: {nm} max |h1 + h2 - month total| = {gap:.1f}")

# The two periods are not equally hard. Say so before comparing any score across them.
tot = con.sql(f"""SELECT strftime(report_date, '%Y-%m') AS mth, SUM(gsc_impressions) AS impr,
                         COUNT(DISTINCT content_hash_id) AS pages
                  FROM read_parquet([{paths}]) WHERE gsc_data_available IS TRUE
                  GROUP BY 1 ORDER BY 1""").df()
tot["impr_M"] = (tot["impr"] / 1e6).round(1)
print("\nPortfolio level context. The label months are not equally kind:")
print(tot[["mth", "impr_M", "pages"]].to_string(index=False))
print("  April rose against March and May fell against April, so the two panels sit in different")
print("  portfolio conditions. Whether that arrives as a larger share of declining pages is a separate")
print(f"  question, and the answer is no: panel 1 base rate {P1['decline'].mean():.4f}, panel 2 "
      f"{P2['decline'].mean():.4f}. The May fall is concentrated rather than broad, so it shows up in")
print("  the impression total without lifting the share of pages that crossed the 0.8 threshold.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PANEL 1  features 2026-03, label 2026-04, decision day 2026-03-31
  pages 176,738   clients 47   base decline rate 0.5319
PANEL 2  features 2026-04, label 2026-05, decision day 2026-04-30
  pages 194,760   clients 51   base decline rate 0.5030

check: panel 1 reproduces the ML-08 frame exactly, 176,738 pages at base rate 0.5319.
check: panel 1 max |h1 + h2 - month total| = 0.0
check: panel 2 max |h1 + h2 - month total| = 0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Portfolio level context. The label months are not equally kind:
    mth  impr_M  pages
2026-03   280.7 176738
2026-04   292.1 194760
2026-05   266.2 237910
  April rose against March and May fell against April, so the two panels sit in different
  portfolio conditions. Whether that arrives as a larger share of declining pages is a separate
  question, and the answer is no: panel 1 base rate 0.5319, panel 2 0.5030. The May fall is concentrated rather than broad, so it shows up in
  the impression total without lifting the share of pages that crossed the 0.8 threshold.


In [3]:
# Numbers behind my second question about the paper, tested on my own slice.
# The question was whether an age gradient measured across pages can be read as decay within a page.
AGE_BINS = [-1, 30, 90, 180, 365, 10**9]
AGE_LBL  = ["0-30", "30-90", "90-180", "180-365", "365+"]

# (a) Survivorship. How much of the catalogue does an "active pages" filter keep, by age?
cat = dc.copy()
cat["age_days"] = (pd.Timestamp("2026-03-31") - cat["content_created_date"]).dt.days
cat = cat[cat["age_days"].notna()]
cat["kept"] = cat["content_hash_id"].isin(set(P1["content_hash_id"]))
cat["band"] = pd.cut(cat["age_days"], AGE_BINS, labels=AGE_LBL)
s = cat.groupby("band", observed=True)["kept"].agg(catalogued="size", kept="sum")
s["kept_share"] = (s["kept"] / s["catalogued"]).round(3)
print("(a) An active pages filter is not age neutral. My slice keeps 176,738 of "
      f"{len(cat):,} catalogued pages:")
print(s.to_string())

# (b) Snapshot level versus within page trajectory, same rows, same bands.
tr = P1.merge(w[["content_hash_id", "may_impr"]], on="content_hash_id", how="left")
tr["may_impr"] = tr["may_impr"].fillna(0)
tr["band"] = pd.cut(tr["age_days"], AGE_BINS, labels=AGE_LBL)
tr["apr_over_mar"] = (tr["next_impr"] + 1) / (tr["impr"] + 1)
tr["may_over_mar"] = (tr["may_impr"] + 1) / (tr["impr"] + 1)
g = tr.groupby("band", observed=True).agg(
    n=("impr", "size"),
    median_march_impr=("impr", "median"),
    decline_rate=("decline", "mean"),
    median_apr_over_mar=("apr_over_mar", "median"),
    median_may_over_mar=("may_over_mar", "median"),
).round(3)
print("\n(b) Cross sectional level next to the two month trajectory of the same pages:")
print(g.to_string())
print("\n  'median_march_impr' is the snapshot read, one number per age band across different pages.")
print("  'median_may_over_mar' is the trajectory read, each page against its own March baseline.")

(a) An active pages filter is not age neutral. My slice keeps 176,738 of 519,606 catalogued pages:
         catalogued   kept  kept_share
band                                  
0-30          27033  16372       0.606
30-90         51854  41363       0.798
90-180        41630  26247       0.630
180-365      210416  71046       0.338
365+         102501  21710       0.212



(b) Cross sectional level next to the two month trajectory of the same pages:
             n  median_march_impr  decline_rate  median_apr_over_mar  median_may_over_mar
band                                                                                     
0-30     16372              118.0         0.264                1.607                0.805
30-90    41363              208.0         0.500                0.800                0.528
90-180   26247              239.0         0.583                0.700                0.500
180-365  71046              128.0         0.594                0.667                0.556
365+     21710              244.0         0.528                0.763                0.711

  'median_march_impr' is the snapshot read, one number per age band across different pages.
  'median_may_over_mar' is the trajectory read, each page against its own March baseline.


### What my slice says about that second question

Both checks came back sharper than I expected, and one of them came back pointing the other way.

**The active pages filter is strongly age selective.** It keeps about 80 percent of pages aged 30 to 90
days and about 21 percent of pages past a year. The oldest band in my slice is therefore a fifth of the
oldest band in the catalogue, and the four fifths that dropped out are the ones with no search
visibility at all. Anything I plot against age is plotted across populations that were filtered at very
different rates, which is the same structural property the paper's active content subset has.

**That selection is enough to erase the decay pattern in my slice, and it erases it in the direction
that shows how much work the filter does.** Median March impressions do not fall with age here. The
oldest band is the highest of the five, 244 against 118 for the youngest. Measured against their own
earlier selves the oldest pages also hold up second best of the five bands. If I read my own snapshot
the way finding #2 reads its snapshot, I would announce that content gets stronger after a year, which
I do not believe either. My curve and the paper's curve disagree because the two filters differ, and
that is the whole point: neither one is measuring what happens to a page as it ages.

**One gradient does survive, and it still is not decay.** Decline rate climbs from 0.264 in the
youngest band to roughly 0.59 by 180 to 365 days, then eases at 365+. That is a real ordering in my
data, but it is a comparison between different pages, and the easing at the oldest band sits exactly
where survivorship is strongest, so I would not read the final point as a rebound.

This is one portfolio over one quarter and a different slice from the paper's, so it corroborates the
methodology question rather than correcting the paper's numbers. What it does settle for my own work is
that I cannot use an age gradient measured across pages as evidence about the life of a page, which is
directly relevant because my Week 4 rule ranked on nothing but age.

## 2. My model under an honest split (before/after)

Week 5 already grouped by client, so simply repeating that would not audit anything. The gap Week 5
left open was time: features came from March and the label from April, but the model was never asked
to work in a period it had not seen. Every number in ML-08 came from one month pair. So the
improvement I test here is a **time aware forward split**, and I put it on a ladder with the splits
Week 5 used so the before and after sit in one table.

Four designs, same random forest, same ten features, same frozen Week 4 rule alongside it:

| rung | design | what the test row is allowed to share with training |
|---|---|---|
| 1 | random split, March to April | same clients, same period |
| 2 | client grouped folds, March to April (ML-08) | same period only |
| 3 | time forward, train March to April, test April to May | same clients only |
| 4 | grouped and time forward | nothing |

Rung 4 is the one that matches deployment: a client FlyRank has not worked with, in a month that has
not happened yet. Rungs 1, 2 and 4 report the mean and spread over five repeats. Rung 3 is a single
fit on the whole panel, so it has no spread, and I mark it rather than pretending otherwise.

One thing to hold onto while reading the table: the two periods are not equally hard. April impressions
rose against March and May fell against April, so panel 1 sits in a rising portfolio and panel 2 in a
falling one. A score that moves between rung 2 and rung 3 could be the model failing to travel or the
period behaving differently, and one table cannot separate those. That is why every rung prints the base
rate of its own test set, and it is worth checking whether the aggregate fall actually arrives as a
larger share of declining pages before assuming it does.

In [4]:
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

STATIC = ["impr", "clicks", "ctr", "avg_position", "days_with_impr", "age_days"]
MOMENT = ["mom_impr", "mom_clicks", "h1_impr", "h2_impr"]
FEATS  = STATIC + MOMENT      # the same ten columns as ML-08, renamed from mar_* to a neutral prefix

def fit_rf(X, y):
    """The exact ML-08 model. Nothing retuned for this notebook."""
    return RandomForestClassifier(n_estimators=200, min_samples_leaf=100,
                                  n_jobs=-1, random_state=42).fit(X, y)

def baseline_w04(df):
    """My frozen ML-07 rule, rescored on whatever rows it is given."""
    flag = (df["impr"] >= 50) & (df["age_days"] >= 90)
    return np.where(flag, df["age_days"], 0).astype(float)

def p_at_k(scores, yy, k, tiebreak=None):
    tb = tiebreak if tiebreak is not None else np.zeros(len(scores))
    order = np.lexsort((-np.asarray(tb), -np.asarray(scores)))
    return np.asarray(yy)[order[:k]].mean()

def row(design, rep, model, scores, yy, tiebreak=None, shared=""):
    return {"design": design, "rep": rep, "model": model,
            "AUC": roc_auc_score(yy, scores),
            "PR_AUC": average_precision_score(yy, scores),
            "p@50": p_at_k(scores, yy, 50, tiebreak),
            "p@500": p_at_k(scores, yy, 500, tiebreak),
            "test_base": float(np.mean(yy)), "shared_clients": shared}

X1, y1, g1 = P1[FEATS].fillna(0.0), P1["decline"].values, P1["client_hash_id"].values
X2, y2, g2 = P2[FEATS].fillna(0.0), P2["decline"].values, P2["client_hash_id"].values
gkf = GroupKFold(n_splits=5)
L = []

# rung 1: random split inside one period. Clients appear on both sides.
D = "1 random, Mar>Apr"
for s in range(5):
    tr, te = train_test_split(np.arange(len(P1)), test_size=0.2, stratify=y1, random_state=s)
    p = fit_rf(X1.iloc[tr], y1[tr]).predict_proba(X1.iloc[te])[:, 1]
    sh = f"{len(set(g1[te]) & set(g1[tr]))} of {len(set(g1[te]))}"
    L.append(row(D, s, "rf_momentum", p, y1[te], shared=sh))
    L.append(row(D, s, "baseline_w04", baseline_w04(P1.iloc[te]), y1[te], P1["impr"].values[te], sh))
print("rung 1 done")

# rung 2: client grouped folds inside one period. This is the ML-08 design.
D = "2 grouped, Mar>Apr"
for f, (tr, te) in enumerate(gkf.split(X1, y1, g1)):
    p = fit_rf(X1.iloc[tr], y1[tr]).predict_proba(X1.iloc[te])[:, 1]
    sh = f"{len(set(g1[te]) & set(g1[tr]))} of {len(set(g1[te]))}"
    L.append(row(D, f, "rf_momentum", p, y1[te], shared=sh))
    L.append(row(D, f, "baseline_w04", baseline_w04(P1.iloc[te]), y1[te], P1["impr"].values[te], sh))
print("rung 2 done")

# rung 3: forward in time. Train on the whole March to April panel, test on the whole April to May
# panel. Clients are shared, the period is not. Single fit, so no spread.
D = "3 time fwd, Apr>May"
p = fit_rf(X1, y1).predict_proba(X2)[:, 1]
sh = f"{len(set(g2) & set(g1))} of {len(set(g2))}"
L.append(row(D, 0, "rf_momentum", p, y2, shared=sh))
L.append(row(D, 0, "baseline_w04", baseline_w04(P2), y2, P2["impr"].values, sh))
print("rung 3 done")

# rung 4: grouped AND forward. Train on held in clients in the earlier pair, test on held out
# clients in the later pair. Unseen client, unseen month. This is the deployment question.
D = "4 grouped+time fwd"
for f, (tr, te) in enumerate(gkf.split(X1, y1, g1)):
    mask = np.isin(g2, list(set(g1[te])))
    if mask.sum() < 500:
        continue
    mdl = fit_rf(X1.iloc[tr], y1[tr])
    p = mdl.predict_proba(X2[mask])[:, 1]
    sh = f"{len(set(g2[mask]) & set(g1[tr]))} of {len(set(g2[mask]))}"
    L.append(row(D, f, "rf_momentum", p, y2[mask], shared=sh))
    L.append(row(D, f, "baseline_w04", baseline_w04(P2[mask]), y2[mask], P2["impr"].values[mask], sh))
print("rung 4 done")

lad = pd.DataFrame(L)
num = ["AUC", "PR_AUC", "p@50", "p@500", "test_base"]
mean_l = lad.groupby(["design", "model"])[num].mean().round(3)
std_l  = lad.groupby(["design", "model"])[num].std().round(3)
reps   = lad.groupby(["design", "model"]).size().rename("reps")

print("\nBEFORE AND AFTER, the same model and the same frozen rule under four splits")
print(mean_l.join(reps).to_string())
print("\nSpread across repeats (standard deviation, blank where there is only one fit)")
print(std_l.to_string())
print("\nWhat each test row was allowed to share with training:")
print(lad.groupby("design")["shared_clients"].first().to_string())

rung 1 done


rung 2 done


rung 3 done


rung 4 done

BEFORE AND AFTER, the same model and the same frozen rule under four splits
                                    AUC  PR_AUC   p@50  p@500  test_base  reps
design              model                                                     
1 random, Mar>Apr   baseline_w04  0.513   0.532  0.968  0.623      0.532     5
                    rf_momentum   0.763   0.773  0.972  0.938      0.532     5
2 grouped, Mar>Apr  baseline_w04  0.532   0.579  0.568  0.622      0.532     5
                    rf_momentum   0.671   0.673  0.892  0.774      0.532     5
3 time fwd, Apr>May baseline_w04  0.509   0.483  0.580  0.578      0.503     1
                    rf_momentum   0.690   0.698  0.980  0.972      0.503     1
4 grouped+time fwd  baseline_w04  0.512   0.507  0.324  0.383      0.509     5
                    rf_momentum   0.679   0.673  0.916  0.867      0.509     5

Spread across repeats (standard deviation, blank where there is only one fit)
                                    AUC  P

### Reading the before and after

Six readings of that table, and two of them are the reverse of what I assumed when I planned this
section.

**The random to grouped step reproduces Week 5.** Rung 1 to rung 2 is the before and after ML-08 already
reported and it lands the same way here: the forest's AUC falls 0.763 to 0.671 and the rule's
precision@50 falls 0.968 to 0.568 as soon as a client can no longer sit on both sides of the split.
Rung 1 was memorising client habits. It is also the rung that would look best on a slide, which is the
whole reason to keep rung 2 next to it.

**The forest travels forward almost unchanged.** AUC 0.671 grouped, 0.690 time forward, 0.679 with both
constraints at once. Whatever it read out of March is worth about the same in a month it was not trained
on, for clients it has never seen. I want to be careful with that: the repeat standard deviation on those
rungs is around 0.06, so 0.671 and 0.679 are not distinguishable. The claim is that the strict design did
not cost the forest measurable AUC, not that it helped.

**The frozen rule does not travel, and the top of its queue is where it breaks.** Its AUC sits between
0.51 and 0.53 on every rung, which is the coin flip it always was outside rung 1. What moves is the head of the queue:
precision@50 runs 0.968, 0.568, 0.580, then 0.324 at rung 4 against that test set's base rate of 0.509.
At rung 4 the fifty pages the rule is most certain about decline less often than fifty pages drawn at
random. Ranking on age was not weakly informative there, it was pointing the wrong way.

**So the gap between model and rule widens under the strictest design.** Rung 2: 0.671 against 0.532 on
AUC, 0.892 against 0.568 at precision@50. Rung 4: 0.679 against 0.512, 0.916 against 0.324. I expected
the gap to shrink and it did the opposite. It is also the result I would hedge hardest, because the rule's
rung 4 precision@50 carries a standard deviation of 0.187 across repeats, and with 47 clients split five
ways each fold is scoring on the order of ten held out clients.

**Precision@K is not comparable across rungs and I nearly read it that way.** Rung 3 scores the whole of
panel 2, 194,760 rows, while rungs 2 and 4 score roughly 35,000 and 39,514. Taking the fifty best out of a
much larger pool is an easier task, so rung 3's 0.980 is not evidence that the model does better there
than rung 4's 0.916. Precision@K is readable down a column only when the pools are the same size. AUC does
not have that problem, so I read AUC across rungs and precision only against the base rate printed beside
it.

**Rung 3 stays confounded.** It holds clients constant and changes only the period, so in principle it
isolates travelling through time, but the period itself moved: the portfolio fell in May having risen in
April. The fall does not arrive as more pages declining, panel 2's base rate is 0.503 against panel 1's
0.532, so the aggregate drop is concentrated in fewer, larger pages rather than spread across the
catalogue. Useful to know, and it does not resolve the confound. Two or three more month pairs would let
the period effect average out, and that is the obvious next thing to run.

## 3. Leakage audit

Four tests on the final ten column feature set, in the order the leakage taxonomy suggests. The first
one is a control on the harness itself: if I deliberately hand the model a piece of the answer and the
score does not jump, then my test rig cannot detect leakage and none of the other results mean
anything.

In [5]:
tr0, te0 = next(gkf.split(X1, y1, g1))     # one grouped fold, held fixed across every test below

def auc_on(cols_or_X, panel=P1, tr=tr0, te=te0, yy=y1):
    Xz = panel[cols_or_X].fillna(0.0) if isinstance(cols_or_X, list) else cols_or_X.fillna(0.0)
    return roc_auc_score(yy[te], fit_rf(Xz.iloc[tr], yy[tr]).predict_proba(Xz.iloc[te])[:, 1])

# A. Positive control. Hand the model the label's own ingredient and confirm the rig reacts.
Xleak = X1.copy()
Xleak["april_impressions_LEAK"] = P1["next_impr"].values
auc_leak, auc_real = auc_on(Xleak), auc_on(FEATS)
print("A. POSITIVE CONTROL, does my harness notice leakage at all")
print(f"   ten honest features                      AUC {auc_real:.4f}")
print(f"   plus the April impressions the label uses AUC {auc_leak:.4f}")
print(f"   the rig moves {auc_leak - auc_real:+.4f} when handed the answer, so it can detect leakage.")

# B. The suspect. mom_impr dominated permutation importance in ML-08, which is the shape leakage makes.
auc_no_mom  = auc_on([c for c in FEATS if c != "mom_impr"])
auc_static  = auc_on(STATIC)
print("\nB. THE SUSPECT, drop momentum and see whether the model collapses or just gets worse")
print(f"   all ten features        AUC {auc_real:.4f}")
print(f"   without mom_impr        AUC {auc_no_mom:.4f}   ({auc_no_mom - auc_real:+.4f})")
print(f"   static six only         AUC {auc_static:.4f}   ({auc_static - auc_real:+.4f})")
print("   Leakage looks like a fall from near 1.0. This is a fall from a modest number to a")
print("   weaker one, which is what a genuinely useful feature looks like, not a label in disguise.")

# C. Arithmetic coupling. This is the caveat I wrote into ML-08 and never tested.
# decline is (April < 0.8 x March total), and March total contains h1, which momentum divides by.
# So a page that slid inside March has an inflated denominator. Test it on a label that shares
# no term with March at all: May against April, features still from March only.
d = P1.merge(w[["content_hash_id", "may_impr"]], on="content_hash_id", how="left")
d["may_impr"] = d["may_impr"].fillna(0)
d = d[d["next_impr"] > 0].reset_index(drop=True)
d["decline_decoupled"] = (d["may_impr"] < 0.8 * d["next_impr"]).astype(int)
Xd, yd, gd = d[FEATS].fillna(0.0), d["decline_decoupled"].values, d["client_hash_id"].values
tr1, te1 = next(GroupKFold(n_splits=5).split(Xd, yd, gd))
auc_dec = roc_auc_score(yd[te1], fit_rf(Xd.iloc[tr1], yd[tr1]).predict_proba(Xd.iloc[te1])[:, 1])
print("\nC. ARITHMETIC COUPLING, does March momentum still work on a label that shares no term with March")
print(f"   pages {len(d):,}   decoupled base rate {yd.mean():.4f}")
print(f"   March features, April vs March label (shares the March total)  AUC {auc_real:.4f}")
print(f"   March features, May vs April label   (shares nothing)          AUC {auc_dec:.4f}")
q = pd.qcut(d["mom_impr"].rank(method="first"), 4,
            labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])   # rank first, mom_impr has heavy ties
cmp_tbl = pd.DataFrame({
    "n": d.groupby(q, observed=True).size(),
    "orig_label_decline": d.groupby(q, observed=True)["decline"].mean(),
    "decoupled_label_decline": d.groupby(q, observed=True)["decline_decoupled"].mean(),
}).round(3)
print("   Decline rate by March momentum quartile, under both labels:")
print(cmp_tbl.to_string())

# D. The three columns I excluded, re measured rather than asserted.
fb = con.sql(f"""SELECT COUNT(*) AS catalogued,
       COUNT(*) FILTER (WHERE content_updated_date       > DATE '2026-03-31') AS upd_future,
       COUNT(*) FILTER (WHERE last_optimized_date        IS NOT NULL)         AS opt_present,
       COUNT(*) FILTER (WHERE last_optimized_date        > DATE '2026-03-31') AS opt_future,
       COUNT(*) FILTER (WHERE optimization_eligible_date IS NOT NULL)         AS elig_present,
       COUNT(*) FILTER (WHERE optimization_eligible_date > DATE '2026-03-31') AS elig_future
       FROM read_parquet('{DIMC}')""").df().iloc[0]
print("\nD. THE EXCLUDED COLUMNS, still future information at my 2026-03-31 decision day")
print(f"   content_updated_date       {fb.upd_future:,} of {fb.catalogued:,} pages dated after the decision day "
      f"({100*fb.upd_future/fb.catalogued:.1f}%)")
print(f"   last_optimized_date        {fb.opt_future:,} of {fb.opt_present:,} non null values after it "
      f"({100*fb.opt_future/max(fb.opt_present,1):.0f}%)")
print(f"   optimization_eligible_date {fb.elig_future:,} of {fb.elig_present:,} non null values after it "
      f"({100*fb.elig_future/max(fb.elig_present,1):.0f}%)")
print("   Filtering to pre decision values does not rescue content_updated_date: the dimension holds")
print("   one snapshot value per page, so a page updated in May has overwritten whatever its March")
print("   era update date was. The March value is not recoverable, so the column stays out.")

# E. Population selection, disclosed rather than buried.
print("\nE. POPULATION, what my slice keeps and what that costs")
print(f"   catalogued pages {len(dc):,}")
print(f"   panel 1 keeps {len(P1):,} ({100*len(P1)/len(dc):.1f}%), panel 2 keeps {len(P2):,}")
print("   filter: gsc_data_available IS TRUE and at least one impression in the feature month.")
print("   That is a survivor filter, and section 1(a) measured that it is not age neutral.")
print("   Every number in this notebook describes pages that were already visible in search.")

A. POSITIVE CONTROL, does my harness notice leakage at all
   ten honest features                      AUC 0.6428
   plus the April impressions the label uses AUC 0.9749
   the rig moves +0.3321 when handed the answer, so it can detect leakage.



B. THE SUSPECT, drop momentum and see whether the model collapses or just gets worse
   all ten features        AUC 0.6428
   without mom_impr        AUC 0.6159   (-0.0269)
   static six only         AUC 0.5802   (-0.0626)
   Leakage looks like a fall from near 1.0. This is a fall from a modest number to a
   weaker one, which is what a genuinely useful feature looks like, not a label in disguise.



C. ARITHMETIC COUPLING, does March momentum still work on a label that shares no term with March
   pages 158,549   decoupled base rate 0.5433
   March features, April vs March label (shares the March total)  AUC 0.6428
   March features, May vs April label   (shares nothing)          AUC 0.6610
   Decline rate by March momentum quartile, under both labels:
                n  orig_label_decline  decoupled_label_decline
mom_impr                                                      
Q1 lowest   39638               0.677                    0.476
Q2          39637               0.548                    0.517
Q3          39637               0.401                    0.547
Q4 highest  39637               0.287                    0.632


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


D. THE EXCLUDED COLUMNS, still future information at my 2026-03-31 decision day
   content_updated_date       382,739 of 519,606 pages dated after the decision day (73.7%)
   last_optimized_date        45,396 of 45,396 non null values after it (100%)
   optimization_eligible_date 45,396 of 45,396 non null values after it (100%)
   Filtering to pre decision values does not rescue content_updated_date: the dimension holds
   one snapshot value per page, so a page updated in May has overwritten whatever its March
   era update date was. The March value is not recoverable, so the column stays out.

E. POPULATION, what my slice keeps and what that costs
   catalogued pages 519,606
   panel 1 keeps 176,738 (34.0%), panel 2 keeps 194,760
   filter: gsc_data_available IS TRUE and at least one impression in the feature month.
   That is a survivor filter, and section 1(a) measured that it is not age neutral.
   Every number in this notebook describes pages that were already visible in search.


### Leakage verdict

**The harness works.** Test A is the control that licenses everything after it. Ten honest features score
0.6428 on a fixed grouped fold; adding the April impression total the label is computed from takes the same
fold to 0.9749. The rig moves 33 AUC points when handed the answer, so a leaked column here announces
itself rather than passing quietly.

**Momentum is not a label in disguise.** Test B rules out the crude failure. Dropping `mom_impr` takes
0.6428 to 0.6159, and the static six alone reach 0.5802, so the momentum block is worth about six AUC
points and `mom_impr` under three of them on the margin. That is the shape of a useful feature.
A leak, as test A just demonstrated in the same rig, moves the score 33 points and lands near 0.97.

**Test C is the one I owed from Week 5, and it changed what I think momentum measures.** The decline label
divides by the March total, and the March total contains the first half of March that momentum divides by,
so a page that slid inside March carries an inflated denominator and clears the 0.8 threshold more easily.
Rebuilt on a label that shares no term with March at all, May against April, the AUC does not fall, it
rises slightly, 0.6428 to 0.6610. Read alone that looks like a clean pass. The quartile table underneath
says otherwise. Under the original label the decline rate falls monotonically across March momentum
quartiles: 0.677, 0.548, 0.401, 0.287. Under the decoupled label it rises: 0.476, 0.517, 0.547, 0.632.
The direction inverts.

**What that licenses me to say.** Inside the month pair that shares a denominator, a page sliding in March
was associated with declining into April. One month further out the same slide is associated with
declining less often than a page that was climbing, which is the shape of mean reversion and also exactly
what the arithmetic coupling would manufacture on the near label. The two are not separable with the
design I have. What is clear is why the AUC held up across the flip: the forest is free to learn whichever
direction its training data shows, so a stable AUC says the feature is informative about something, not
that my story about it is right. `mom_impr` stays in the model as a ranking input. My Week 5 sentence about
a sliding page continuing to slide does not survive, and it is rewritten in section 4.

**The excluded columns are still correctly excluded, and one is worse than Week 3 recorded.**
`last_optimized_date` and `optimization_eligible_date` are post decision on 100 percent of their non null
values, which is straightforward. `content_updated_date` is the subtle one: 73.7 percent of catalogued
pages carry a value after my decision day, and the dimension holds a single snapshot per page, so for those
pages the March era value has been overwritten and is not recoverable. A pre decision filter does not fix
that, it just quietly keeps the pages nobody touched. Excluded, now for a reason I can state precisely
rather than by rule of thumb.

**The population is a survivor sample and that is on the record.** My slice is pages that already had
search visibility in the feature month, 34 percent of the catalogue, and section 1 measured that the filter
is not age neutral. Every number in this notebook describes already visible pages.

In [6]:
# Real pages the model gets wrong, from the strictest design that still has enough test rows.
# Ids are truncated warehouse hashes. No URLs, no client names, no queries.
mask4 = np.isin(g2, list(set(g1[te0])))
mdl4  = fit_rf(X1.iloc[tr0], y1[tr0])
ex = P2[mask4].copy()
ex["score"] = mdl4.predict_proba(X2[mask4])[:, 1]
ex["page"]  = ex["content_hash_id"].astype(str).str[:10]
show = ["page", "score", "decline", "impr", "h1_impr", "h2_impr", "mom_impr",
        "avg_position", "age_days", "next_impr"]

print(f"Unseen client, unseen month. {len(ex):,} test pages, base decline rate {ex['decline'].mean():.3f}, "
      f"model AUC {roc_auc_score(ex['decline'], ex['score']):.3f}\n")

print("FALSE POSITIVES, the five the model was most confident about that did NOT decline")
print(ex[ex["decline"] == 0].nlargest(5, "score")[show].round(3).to_string(index=False))

print("\nFALSE NEGATIVES, the five the model was least worried about that DID decline")
print(ex[ex["decline"] == 1].nsmallest(5, "score")[show].round(3).to_string(index=False))

print("\nShape of each error group against the rows it got right")
flagged = ex["score"].values >= 0.5
declined = ex["decline"].values == 1
ex["outcome"] = np.select(
    [flagged & declined, flagged & ~declined, ~flagged & declined],
    ["true positive", "false positive", "false negative"], default="true negative")
print(ex.groupby("outcome").agg(n=("impr", "size"), median_impr=("impr", "median"),
                                median_mom=("mom_impr", "median"), median_age=("age_days", "median"),
                                median_pos=("avg_position", "median")).round(2).to_string())

# The false negatives all look alike, and the reason is in the feature definition rather than the data.
new_half = ex[(ex["h1_impr"] == 0) & (ex["h2_impr"] > 0)]
print("\nWHY THE FALSE NEGATIVES LOOK ALIKE")
print("All five have h1_impr 0 and an age near 18 days, so mom_impr = (h2+1)/(h1+1) collapses to h2+1.")
print("For those rows the feature is not a ratio at all, it is a level, and a large value means the page")
print("first appeared after the 15th rather than that it is gaining.")
print(f"   pages with no first half impressions {len(new_half):,} of {len(ex):,} "
      f"({100*len(new_half)/len(ex):.1f}%)")
print(f"   of those, decline rate {new_half['decline'].mean():.3f} against {ex['decline'].mean():.3f} overall, "
      f"median model score {new_half['score'].median():.3f}")
print("   A month over month label on a page that only existed for half of the feature month is a")
print("   comparison between a partial month and a full one, so these rows are mislabelled by design.")

# Calibration at the default threshold, which is the number that decides how the queue is used.
n_flag, n_dec = int(flagged.sum()), int(declined.sum())
acc_all = float(((ex["score"] >= 0.5).astype(int) == ex["decline"]).mean())
print("\nCALIBRATION AT THE DEFAULT 0.5 THRESHOLD")
print(f"   flagged {n_flag:,} of {len(ex):,} rows ({100*n_flag/len(ex):.1f}%) while {n_dec:,} "
      f"({100*n_dec/len(ex):.1f}%) actually declined")
print(f"   accuracy {acc_all:.3f}, and the majority class alone would score {max(1-ex['decline'].mean(), ex['decline'].mean()):.3f}")
print("   The model over flags heavily at 0.5, so the score is a usable ordering and not a calibrated")
print("   probability. A deployment should set the cut from how many pages a reviewer can actually open,")
print("   which is what precision@50 measures, not from the library default.")

thin = ex[ex["impr"] < 150]
acc_thin = float(((thin["score"] >= 0.5).astype(int) == thin["decline"]).mean())
print("\nWHERE THE LABEL ITSELF IS WEAK, which is a different problem from a wrong prediction")
print(f"   pages under 150 impressions {len(thin):,} of {len(ex):,} ({100*len(thin)/len(ex):.1f}%)")
print("   A 20 percent month over month move on 150 impressions is about 30 impressions, inside ordinary")
print("   traffic noise, so on a share of these rows the label records noise rather than an outcome.")
print(f"   Accuracy on that band is {acc_thin:.3f} against {acc_all:.3f} overall, slightly higher, which is")
print("   the point: accuracy cannot tell me whether the thing I labelled was worth predicting.")

Unseen client, unseen month. 39,514 test pages, base decline rate 0.423, model AUC 0.601

FALSE POSITIVES, the five the model was most confident about that did NOT decline
      page  score  decline   impr  h1_impr  h2_impr  mom_impr  avg_position  age_days  next_impr
content_6e  0.954        0 2432.0   1786.0    646.0     0.362         4.821       273     2259.0
content_ae  0.953        0 3174.0   2340.0    834.0     0.357         4.183       196     2721.0
content_8b  0.952        0 2482.0   1963.0    519.0     0.265         3.208        84     2621.0
content_2c  0.950        0  656.0    389.0    267.0     0.687        18.131       442      822.0
content_14  0.949        0  661.0    391.0    270.0     0.691        18.221       423     1097.0

FALSE NEGATIVES, the five the model was least worried about that DID decline
      page  score  decline  impr  h1_impr  h2_impr  mom_impr  avg_position  age_days  next_impr
content_ef  0.045        1 233.0      0.0    233.0     234.0         6.2

## 4. Claim rewrite

Four sentences I wrote in Weeks 4 and 5 that went further than the design supports, with what the evidence
actually carries and the version I would now put in front of a client. Three of them share one fault: I
wrote a property of pages when what I had measured was a property of my sample and my period. The fourth
is worse than that. It reverses when the label moves a month.

**1. Week 4, on the top of the baseline queue**

> Before: "precision@50 of 0.960, the rule puts declining pages at the top of the queue."

That 0.960 was measured with every client pooled, and my own weak picks check found the top 100 came
from a single client. Rescored on unseen clients it lands near 0.57, and the stratified dummy reaches
about the same. The number was mostly describing one client's March.

> After: "Across five client grouped folds, the Week 4 rule's precision@50 averaged 0.57 against a base
> rate of 0.53, with fold to fold values from 0.14 to 0.96. The pooled 0.960 reported in Week 4 was
> driven by one client that dominated the top of the queue and should not be quoted as the rule's
> performance."

**2. Week 5, on momentum**

> Before: "Momentum is what actually moves the number. A page already sliding in March tends to keep
> sliding."

The first sentence is a fair description of my table. The second is an interpretation I attached to it, and
section 3 test C says it does not hold one month further out: across March momentum quartiles the decline
rate falls under the April label and rises under the decoupled May label. The direction inverts, so the
interpretation goes.

> After: "In the March to April panel, adding within month momentum was associated with the largest single
> improvement in the comparison, on every metric measured. The direction of that association is period
> specific: pages sliding within March declined into April more often than pages climbing (0.677 in the
> lowest momentum quartile against 0.287 in the highest), while against a May over April label the ordering
> reverses (0.476 rising to 0.632). Part of the near label association comes from the label sharing the
> March total with the feature. Momentum is measured as informative for ranking one month ahead in this
> panel. It is not evidence that decline persists."

**3. Week 5, on the model beating the rule**

> Before: "rf_momentum beats the frozen ML-07 rule on the same grouped folds."

Correct for the split it names, and that split shares the period. Section 2 ran the stricter one expecting
the gap to shrink. It widened.

> After: "Under client grouped folds within one month pair, the forest's mean AUC was 0.671 against the
> Week 4 rule's 0.532 on the same rows. Under an unseen client in an unseen month the observed gap was
> wider, 0.679 against 0.512 on AUC and 0.916 against 0.324 at precision@50, where the rule's top fifty
> declined less often than that test set's 0.509 base rate. Both are five fold means, with fold to fold
> spread at that rung near 0.06 for the forest and 0.08 to 0.19 for the rule, each fold scoring about ten
> held out clients, so the finding is directional: I would report the direction and not the size. For
> decision support I would describe
> the forest as a better ordering of a review queue on this evidence, not as a reliable predictor of which
> pages decline."

**4. Week 5, on age**

> Before: "age_days is the one input that carries nothing once momentum is in the room."

Measured on one fold, one permutation pass, one panel. A near zero importance means this model did not
use the column here, not that age is uninformative about content.

> After: "In permutation importance on one grouped fold, shuffling `age_days` did not reduce the forest's
> AUC, so the column contributed no measurable signal once within month momentum was present in this panel.
> Section 1 reaches the same column from the other side: in my slice median impressions do not fall with age
> at all, the oldest band is the highest of the five, so age is a weak stand in for a page's stage of life
> here. Age does still order decline rate, and that ordering is a comparison between different pages rather
> than a trajectory through one page's life."

**The language rules I am now holding myself to.** Observed and measured for anything from one sample.
Associated with, never causes or improves, for anything without an intervention. Every precision figure
next to its base rate and its spread. Decision support wording for the queue, since the queue orders a
reviewer's attention and does not forecast an outcome. And the period and the population named in the
sentence, not in a footnote, because both of them turned out to be load bearing.

In [7]:
# Receipts. The audit is the deliverable, so the split ladder and the leakage tests get written out.
rec = {
    "task": "ML-09 w06_validation_audit",
    "lane": "Refresh / Content Opportunity Scoring",
    "paper": "FlyRank, The State of AI-Driven SEO, March 2026",
    "paper_findings_audited": [
        {"finding": "#4 The Freshness Multiplier (p9)",
         "question": "selection: refreshed pages were chosen, so the 57x mixes refreshing with choosing; "
                     "health also contains impressions, so the 3.2x and 57x overlap by construction"},
        {"finding": "#2 The Content Performance Curve (p7)",
         "question": "cross sectional snapshot read as a within page lifecycle; active content filter "
                     "makes the old buckets survivors"},
    ],
    "panels": {
        "panel_1": {"features": "2026-03", "label": "2026-04", "decision_day": "2026-03-31",
                    "pages": int(len(P1)), "clients": int(P1["client_hash_id"].nunique()),
                    "base_rate": round(float(P1["decline"].mean()), 4)},
        "panel_2": {"features": "2026-04", "label": "2026-05", "decision_day": "2026-04-30",
                    "pages": int(len(P2)), "clients": int(P2["client_hash_id"].nunique()),
                    "base_rate": round(float(P2["decline"].mean()), 4)},
    },
    "split_ladder_mean": json.loads(mean_l.reset_index().to_json(orient="records")),
    "split_ladder_std": json.loads(std_l.reset_index().to_json(orient="records")),
    "leakage_audit": {
        "positive_control_auc_with_label_ingredient": round(float(auc_leak), 4),
        "honest_auc_same_fold": round(float(auc_real), 4),
        "auc_without_mom_impr": round(float(auc_no_mom), 4),
        "auc_static_only": round(float(auc_static), 4),
        "auc_decoupled_label_may_vs_apr": round(float(auc_dec), 4),
        "decoupled_base_rate": round(float(yd.mean()), 4),
        "momentum_quartile_direction": json.loads(
            cmp_tbl.reset_index().assign(mom_impr=lambda t: t["mom_impr"].astype(str)).to_json(orient="records")),
        "momentum_finding": "decline rate falls across March momentum quartiles under the April label and "
                            "rises under the decoupled May label; the direction inverts, so a stable AUC "
                            "across the two labels does not vindicate the persistence reading of momentum",
        "excluded_columns": {
            "content_updated_date": f"{int(fb.upd_future)}/{int(fb.catalogued)} dated after 2026-03-31; "
                                    "single snapshot value per page so the March era value is unrecoverable",
            "last_optimized_date": f"{int(fb.opt_future)}/{int(fb.opt_present)} non null values after 2026-03-31",
            "optimization_eligible_date": f"{int(fb.elig_future)}/{int(fb.elig_present)} non null values after 2026-03-31",
        },
        "population_filter": "gsc_data_available IS TRUE and at least one impression in the feature month; "
                             "a survivor filter, measured as not age neutral",
    },
    "claim_language": ["observed", "measured", "associated with", "directional", "decision support"],
}
path = os.path.join(OUT, "w06_validation_audit_metrics.json")
with open(path, "w") as f:
    json.dump(rec, f, indent=2)
print("wrote", path)

# Guards. These are asserts, not prose, so a future edit that breaks them fails the run.
FORBIDDEN = {"next_impr", "decline", "may_impr", "apr_impr",
             "content_updated_date", "last_optimized_date", "optimization_eligible_date"}
assert set(FEATS).isdisjoint(FORBIDDEN), "a label or future column reached the feature list"
assert auc_leak > auc_real + 0.05, "positive control did not react, the harness cannot detect leakage"
for pre in ("mar", "apr"):
    assert (w[f"{pre}_h1_impr"] + w[f"{pre}_h2_impr"] - w[f"{pre}_impr"]).abs().max() == 0
print(f"guards passed: {len(FEATS)} features, all decision time; halves reconcile in both panels;")
print("positive control reacted, so the leakage tests above are meaningful.")

wrote /home/zuko/ml-internship/work/outputs/w06_validation_audit_metrics.json
guards passed: 10 features, all decision time; halves reconcile in both panels;
positive control reacted, so the leakage tests above are meaningful.


## Self-check

- [x] Two paper findings named with the page they sit on, each with a concrete methodology question
      about where the label comes from and what the design can carry, plus a specific suggestion for
      making the claim stronger rather than a verdict on it.
- [x] My own model re-run under a stricter split, with the before and after in one table: random,
      client grouped, time forward, and grouped plus time forward, the same forest and the same frozen
      Week 4 rule on every rung.
- [x] The confound in that comparison disclosed in the same section: the portfolio rose in April and fell
      in May, so the two periods are not equally hard, each rung prints the base rate of its own test set,
      and precision@K is not read across rungs whose test pools differ in size.
- [x] Leakage audit with a positive control first, so the tests that follow are known to be sensitive;
      then the suspect feature with and without, the arithmetic coupling test I owed from Week 5, the
      three excluded date columns re-measured, and the survivor filter disclosed.
- [x] Real failure examples from the strictest design, individual pages with their feature values, the
      shape of each error group, the feature definition that produces every false negative, the
      calibration of the 0.5 threshold, and the thin traffic band where the label itself is unreliable.
- [x] Four of my own claims rewritten, with the evidence that each rewrite is standing on, including one
      that had to be withdrawn rather than softened because test C reversed its direction.
- [x] Claim language: observed, measured, associated with, directional, decision support. No causal
      wording anywhere, every precision figure next to its base rate and spread.
- [x] Runs top to bottom with no errors. No client names, URLs, or raw queries in any output; page ids
      are truncated warehouse hashes.